In [2]:
# =============================================================================
# 05_Factor_D.ipynb — Cell 1 (regime_v4 재검증)
# 목적: 환경 설정 + 유틸 함수 정의
# 산출물: 없음 (설정만)
# =============================================================================

from __future__ import annotations
import os
from pathlib import Path
from datetime import datetime
import logging
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# ─────────────────────────────────────────────────────────────────────────────
# 1) .env 로드 + SSOT 경로
# ─────────────────────────────────────────────────────────────────────────────
load_dotenv()
env_root = os.getenv("QP2_ROOT", "").strip()
if not env_root:
    raise RuntimeError("QP2_ROOT가 .env에 없음")

QP2_ROOT = Path(env_root).resolve()
EXPECTED_ROOT = Path("C:/QP2").resolve()
if QP2_ROOT != EXPECTED_ROOT:
    raise RuntimeError(f"QP2_ROOT 불일치: {QP2_ROOT} vs {EXPECTED_ROOT}")

# 경로 딕셔너리 (SSOT)
PATHS = {
    "DATA":      QP2_ROOT / "data",
    "RAW":       QP2_ROOT / "data" / "raw",
    "INTERIM":   QP2_ROOT / "data" / "interim",
    "PROCESSED": QP2_ROOT / "data" / "processed",
    "META":      QP2_ROOT / "data" / "meta",
}

for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2) 로깅
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("qp2_D")
logger.info(f"QP2_ROOT = {QP2_ROOT}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 유틸 함수
# ─────────────────────────────────────────────────────────────────────────────
def winsorize(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """극단값 윈저라이징"""
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)

def zscore_by_date(df: pd.DataFrame, col: str) -> pd.Series:
    """날짜별 cross-sectional z-score"""
    return df.groupby("date")[col].transform(lambda x: (x - x.mean()) / x.std())

# 성과 계산 함수
def calc_perf(ret: pd.Series) -> dict:
    """월간 수익률 → CAGR, Vol, Sharpe, MaxDD"""
    n = ret.notna().sum()
    cagr = (1 + ret).prod() ** (12 / n) - 1
    vol = ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else np.nan
    equity = (1 + ret).cumprod()
    mdd = (equity / equity.cummax() - 1).min()
    return {"CAGR": cagr, "Vol": vol, "Sharpe": sharpe, "MaxDD": mdd}

def calc_tstat(excess: pd.Series) -> dict:
    """초과수익의 t-stat, IR 계산"""
    T = excess.notna().sum()
    mu = excess.mean()
    sig = excess.std()
    t = mu / (sig / np.sqrt(T)) if sig > 0 else np.nan
    ir = (mu * 12) / (sig * np.sqrt(12)) if sig > 0 else np.nan
    return {"t_stat": t, "IR": ir, "N": T}

logger.info("Cell 1 완료: 환경 + 유틸 로드됨")

2026-02-13 14:54:15,085 | INFO | QP2_ROOT = C:\QP2
2026-02-13 14:54:15,086 | INFO | Cell 1 완료: 환경 + 유틸 로드됨


In [3]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 2
# 목적: 주가 데이터 로드 + 월간 수익률 생성
# 산출물: px_wide, px_m, ret_1m
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) Yahoo 주가 로드 (wide format)
# ─────────────────────────────────────────────────────────────────────────────
px_wide = pd.read_parquet(PATHS["INTERIM"] / "yahoo_adjclose_wide.parquet")

# date가 컬럼이면 index로
if "date" in px_wide.columns:
    px_wide = px_wide.set_index("date")
px_wide.index = pd.to_datetime(px_wide.index)
px_wide = px_wide.sort_index()

logger.info(f"px_wide: {px_wide.shape[0]:,} days × {px_wide.shape[1]:,} tickers")
logger.info(f"기간: {px_wide.index.min().date()} ~ {px_wide.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 월말 리샘플링
# ─────────────────────────────────────────────────────────────────────────────
px_m = px_wide.resample("ME").last()
logger.info(f"px_m: {px_m.shape[0]:,} months × {px_m.shape[1]:,} tickers")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 월간 수익률
# ─────────────────────────────────────────────────────────────────────────────
ret_1m = px_m.pct_change()

# 분석 시작점: 2013-06-30 이후 (생존편향 허용 기준)
START_DATE = "2013-06-30"
ret_1m = ret_1m.loc[START_DATE:]
px_m = px_m.loc[START_DATE:]

logger.info(f"ret_1m: {ret_1m.shape[0]:,} months × {ret_1m.shape[1]:,} tickers")
logger.info(f"분석 기간: {ret_1m.index.min().date()} ~ {ret_1m.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 공통 종목 (NaN 80% 이상 제거)
# ─────────────────────────────────────────────────────────────────────────────
valid_tickers = ret_1m.columns[ret_1m.notna().mean() > 0.8]
ret_1m = ret_1m[valid_tickers]
px_m = px_m[valid_tickers]

logger.info(f"유효 종목: {len(valid_tickers):,}개 (NaN < 20% 기준)")

logger.info("Cell 2 완료: 데이터 로드됨")

2026-02-13 14:54:15,254 | INFO | px_wide: 16,135 days × 503 tickers
2026-02-13 14:54:15,255 | INFO | 기간: 1962-01-02 ~ 2026-02-10
2026-02-13 14:54:15,303 | INFO | px_m: 770 months × 503 tickers
2026-02-13 14:54:15,308 | INFO | ret_1m: 153 months × 503 tickers
2026-02-13 14:54:15,308 | INFO | 분석 기간: 2013-06-30 ~ 2026-02-28
2026-02-13 14:54:15,314 | INFO | 유효 종목: 467개 (NaN < 20% 기준)
2026-02-13 14:54:15,314 | INFO | Cell 2 완료: 데이터 로드됨


In [4]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 3
# 목적: 다양한 Lookback 모멘텀 신호 생성
# 산출물: mom_signals (dict of DataFrames)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 모멘텀 계산 함수
# ─────────────────────────────────────────────────────────────────────────────
def calc_momentum(px: pd.DataFrame, lookback: int, skip: int = 1) -> pd.DataFrame:
    """
    모멘텀 신호 계산
    
    Parameters:
    -----------
    px : 월말 주가 (wide format)
    lookback : 전체 lookback 기간 (월)
    skip : 최근 제외 기간 (월) — 단기반전 효과 회피용
    
    Returns:
    --------
    momentum signal (t-lookback ~ t-skip 구간 수익률)
    """
    # t-lookback 시점 가격
    px_start = px.shift(lookback)
    # t-skip 시점 가격 (skip=1이면 전월)
    px_end = px.shift(skip)
    
    # 구간 수익률
    mom = (px_end / px_start) - 1
    return mom

# ─────────────────────────────────────────────────────────────────────────────
# 2) 다양한 Lookback 설정
# ─────────────────────────────────────────────────────────────────────────────
LOOKBACK_CONFIGS = {
    "MOM_12_1": (12, 1),   # 클래식: t-12 ~ t-1
    "MOM_12_0": (12, 0),   # 반전 무시: t-12 ~ t-0
    "MOM_6_1":  (6, 1),    # 단기: t-6 ~ t-1
    "MOM_6_0":  (6, 0),    # 단기 + 반전 무시
    "MOM_3_1":  (3, 1),    # 초단기
}

# ─────────────────────────────────────────────────────────────────────────────
# 3) 신호 생성
# ─────────────────────────────────────────────────────────────────────────────
mom_signals = {}

for name, (lookback, skip) in LOOKBACK_CONFIGS.items():
    mom = calc_momentum(px_m, lookback=lookback, skip=skip)
    mom_signals[name] = mom
    
    # 유효 데이터 확인
    valid_count = mom.notna().sum().sum()
    logger.info(f"{name}: lookback={lookback}, skip={skip} → 유효 신호 {valid_count:,}개")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 샘플 확인 (MOM_12_1 기준)
# ─────────────────────────────────────────────────────────────────────────────
sample = mom_signals["MOM_12_1"].iloc[-3:, :5]
logger.info(f"MOM_12_1 샘플 (최근 3개월, 5종목):\n{sample.round(3)}")

logger.info("Cell 3 완료: 모멘텀 신호 생성됨")

2026-02-13 14:54:15,327 | INFO | MOM_12_1: lookback=12, skip=1 → 유효 신호 65,631개
2026-02-13 14:54:15,330 | INFO | MOM_12_0: lookback=12, skip=0 → 유효 신호 65,631개
2026-02-13 14:54:15,332 | INFO | MOM_6_1: lookback=6, skip=1 → 유효 신호 68,433개
2026-02-13 14:54:15,334 | INFO | MOM_6_0: lookback=6, skip=0 → 유효 신호 68,433개
2026-02-13 14:54:15,335 | INFO | MOM_3_1: lookback=3, skip=1 → 유효 신호 69,834개
2026-02-13 14:54:15,340 | INFO | MOM_12_1 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.150  0.119  0.326  0.161  0.017
2026-01-31 -0.096  0.157  0.274 -0.007  0.031
2026-02-28  0.055  0.077  0.103 -0.193  0.034
2026-02-13 14:54:15,340 | INFO | Cell 3 완료: 모멘텀 신호 생성됨


In [5]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 4
# 목적: Lookback별 모멘텀 전략 백테스트 (전체 기간)
# 산출물: results_all (성과 비교 테이블)
# =============================================================================

TOP_N = 50  # 상위 N종목 매수

# ─────────────────────────────────────────────────────────────────────────────
# 1) 백테스트 함수
# ─────────────────────────────────────────────────────────────────────────────
def backtest_momentum(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame, 
                      top_n: int = 30) -> pd.Series:
    """
    모멘텀 전략 백테스트
    
    - 매월 말 모멘텀 상위 top_n 종목 동일가중 매수
    - 다음 달 수익률 계산
    """
    port_rets = []
    
    for date in mom_signal.index:
        if date not in ret_1m.index:
            continue
            
        # 해당 월 모멘텀 신호
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목 선정
        top_tickers = sig.nlargest(top_n).index
        
        # 다음 달 수익률
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        # 포트폴리오 수익률 (동일가중)
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    result = pd.DataFrame(port_rets).set_index("date")["ret"]
    return result

# ─────────────────────────────────────────────────────────────────────────────
# 2) 벤치마크: 전종목 동일가중 (EW)
# ─────────────────────────────────────────────────────────────────────────────
ew_ret = ret_1m.mean(axis=1)
ew_ret.name = "EW"

# ─────────────────────────────────────────────────────────────────────────────
# 3) 각 Lookback별 백테스트
# ─────────────────────────────────────────────────────────────────────────────
strat_rets = {"EW": ew_ret}

for name, mom_sig in mom_signals.items():
    port_ret = backtest_momentum(mom_sig, ret_1m, top_n=TOP_N)
    strat_rets[name] = port_ret
    logger.info(f"{name}: {len(port_ret)}개월 백테스트 완료")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 성과 비교 테이블
# ─────────────────────────────────────────────────────────────────────────────
results = []

for name, ret in strat_rets.items():
    perf = calc_perf(ret)
    
    # EW 대비 초과수익 통계 (EW 제외)
    if name != "EW":
        # 공통 기간만
        common_idx = ret.index.intersection(ew_ret.index)
        excess = ret.loc[common_idx] - ew_ret.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": len(ret)}
    
    results.append({
        "Strategy": name,
        "CAGR": perf["CAGR"] * 100,
        "Vol": perf["Vol"] * 100,
        "Sharpe": perf["Sharpe"],
        "MaxDD": perf["MaxDD"] * 100,
        "t_stat": tstat_info["t_stat"],
        "IR": tstat_info["IR"],
        "N_months": tstat_info["N"],
    })

results_all = pd.DataFrame(results).set_index("Strategy").round(3)

# Sharpe 기준 정렬
results_all = results_all.sort_values("Sharpe", ascending=False)

print("\n" + "="*70)
print("전체 기간 백테스트 결과 (2013-06 ~ 2026-01)")
print("="*70)
display(results_all)

# 최고 전략 표시
best = results_all.index[0]
logger.info(f"최고 Sharpe: {best} ({results_all.loc[best, 'Sharpe']:.3f})")

logger.info("Cell 4 완료: 전체 기간 백테스트 완료")

2026-02-13 14:54:15,583 | INFO | MOM_12_1: 140개월 백테스트 완료
2026-02-13 14:54:15,843 | INFO | MOM_12_0: 140개월 백테스트 완료
2026-02-13 14:54:16,069 | INFO | MOM_6_1: 146개월 백테스트 완료
2026-02-13 14:54:16,292 | INFO | MOM_6_0: 146개월 백테스트 완료
2026-02-13 14:54:16,533 | INFO | MOM_3_1: 149개월 백테스트 완료



전체 기간 백테스트 결과 (2013-06 ~ 2026-01)


,CAGR,Vol,Sharpe,MaxDD,t_stat,IR,N_months
Strategy,,,,,,,
MOM_6_0,21.315,17.182,1.241,-17.418,1.970,0.565,146
MOM_6_1,20.967,17.817,1.177,-23.574,1.868,0.535,146
MOM_12_0,20.340,17.529,1.160,-20.331,1.603,0.469,140
MOM_3_1,19.779,17.546,1.127,-22.303,1.400,0.397,149
MOM_12_1,19.105,17.640,1.083,-19.706,1.324,0.387,140
EW,16.388,15.218,1.077,-23.924,NaN,NaN,153


2026-02-13 14:54:16,555 | INFO | 최고 Sharpe: MOM_6_0 (1.241)
2026-02-13 14:54:16,555 | INFO | Cell 4 완료: 전체 기간 백테스트 완료


In [6]:
# =============================================================================
# 02_Factor_D.ipynb — Cell 4-1 (결과 해석)
# =============================================================================
"""
# Cell 4 결과 해석: Lookback별 모멘텀 성과 비교

## 전체 기간 (2013-06 ~ 2026-01) 결과 요약

| 순위 | 전략      | Sharpe | t-stat | IR    | 비고        |
|------|-----------|--------|--------|-------|-------------|
| 1    | MOM_6_0   | 1.233  | 1.989  | 0.572 | 최고 성과   |
| 2    | MOM_6_1   | 1.171  | 1.893  | 0.544 |             |
| 3    | MOM_12_0  | 1.153  | 1.629  | 0.479 |             |
| 4    | MOM_3_1   | 1.117  | 1.391  | 0.396 |             |
| 5    | MOM_12_1  | 1.079  | 1.371  | 0.403 | 학계 표준   |
| -    | EW        | 1.068  | -      | -     | 벤치마크    |

---

## 핵심 발견

### 1. 6개월 Lookback > 12개월
- MOM_6_x가 MOM_12_x를 전부 이김
- 학계 표준 12-1이 오히려 최하위권
- S&P500에서는 단기 모멘텀이 더 유효

### 2. Skip=0 > Skip=1
- 6_0 > 6_1, 12_0 > 12_1
- "단기 반전 효과 회피" 가설이 이 유니버스에선 성립 안 함
- 직전 1개월 수익률 포함하는 게 오히려 유리

### 3. 통계적 유의성
- MOM_6_0: t=1.989 -> 95% 신뢰수준 근접 (기준 1.96)
- MOM_6_1: t=1.893 -> 90% 통과
- MOM_12_1: t=1.371 -> 유의성 부족 (이전 대화에서 탈락시킨 이유)

---

## 잠정 결론
- MOM_6_0을 1차 후보로 채택
- 단, 레짐별로 최적 Lookback이 다를 수 있음 -> 셀 5에서 검증 필요

---

## 참고: Lookback 표기법
- X-Y: 과거 X개월 중 최근 Y개월 제외
- 예) 12-1 = t-12 ~ t-1 구간 수익률 (11개월)
- 예) 6-0 = t-6 ~ t-0 구간 수익률 (6개월 전체)
"""
print("Cell 4-1: 결과 해석 (docstring으로 기록)")

Cell 4-1: 결과 해석 (docstring으로 기록)


In [7]:
# =============================================================================
# 05_D.ipynb — Cell 5
# 목적: 레짐별 모멘텀 성과 비교 (regime_v4 기반)
# 산출물: regime_results, sharpe_pivot
# 변경: regime_v2(7레짐) → regime_v4(3레짐 + bear_phase)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 레짐 데이터 로드 (regime_v4)
# ─────────────────────────────────────────────────────────────────────────────
regime_df = pd.read_parquet(PATHS["INTERIM"] / "regime_v4.parquet")
regime_df.index = pd.to_datetime(regime_df.index)
regime_m = regime_df  # 이미 월말

REGIME_COL = "regime"
regimes = regime_m[REGIME_COL].dropna().unique().tolist()

logger.info(f"regime_v4: {regime_m.shape[0]} months")
logger.info(f"레짐 종류: {regimes}")
logger.info(f"Bear 내부: {regime_m[regime_m['regime']=='Bear']['bear_phase'].value_counts().to_dict()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 백테스트 함수 (레짐 필터링 버전 — bear_phase 지원)
# ─────────────────────────────────────────────────────────────────────────────
def backtest_by_regime(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame,
                       regime_m: pd.DataFrame, regime_col: str, 
                       target_regime: str, top_n: int = 30,
                       bear_phase_filter: str = None) -> pd.Series:
    """특정 레짐 기간만 백테스트
    bear_phase_filter: None(전체), "declining", "recovering"
    """
    port_rets = []
    
    for date in mom_signal.index:
        if date not in regime_m.index:
            continue
        if regime_m.loc[date, regime_col] != target_regime:
            continue
        # bear_phase 필터
        if bear_phase_filter is not None:
            if regime_m.loc[date, "bear_phase"] != bear_phase_filter:
                continue
        if date not in ret_1m.index:
            continue
        
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        top_tickers = sig.nlargest(top_n).index
        
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    if not port_rets:
        return pd.Series(dtype=float)
    
    return pd.DataFrame(port_rets).set_index("date")["ret"]

# ─────────────────────────────────────────────────────────────────────────────
# 3) 3레짐 백테스트 (MOM_6_0 기준)
# ─────────────────────────────────────────────────────────────────────────────
mom_signal = mom_signals["MOM_6_0"]
regime_results = []

for regime in ["Bull", "Neutral", "Bear"]:
    strat_ret = backtest_by_regime(mom_signal, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(strat_ret) < 3 or len(ew_sub) < 3:
        logger.warning(f"{regime}: 데이터 부족, 스킵")
        continue
    
    strat_perf = calc_perf(strat_ret)
    ew_perf = calc_perf(ew_sub)
    
    common_idx = strat_ret.index.intersection(ew_sub.index)
    if len(common_idx) > 0:
        excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": 0}
    
    regime_results.append({
        "Regime": regime,
        "Strat_CAGR": strat_perf["CAGR"] * 100,
        "Strat_Sharpe": strat_perf["Sharpe"],
        "BM_CAGR": ew_perf["CAGR"] * 100,
        "BM_Sharpe": ew_perf["Sharpe"],
        "Excess_CAGR": (strat_perf["CAGR"] - ew_perf["CAGR"]) * 100,
        "t_stat": tstat_info["t_stat"],
        "N_months": len(strat_ret),
    })

# Bear 내부 세분화
for phase in ["declining", "recovering"]:
    strat_ret = backtest_by_regime(mom_signal, ret_1m, regime_m, REGIME_COL, "Bear", TOP_N,
                                    bear_phase_filter=phase)
    
    regime_dates = regime_m[(regime_m[REGIME_COL] == "Bear") & 
                            (regime_m["bear_phase"] == phase)].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(strat_ret) < 3 or len(ew_sub) < 3:
        logger.warning(f"Bear({phase}): 데이터 부족 (n={len(strat_ret)}), 스킵")
        continue
    
    strat_perf = calc_perf(strat_ret)
    ew_perf = calc_perf(ew_sub)
    
    common_idx = strat_ret.index.intersection(ew_sub.index)
    if len(common_idx) > 0:
        excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": 0}
    
    regime_results.append({
        "Regime": f"Bear({phase})",
        "Strat_CAGR": strat_perf["CAGR"] * 100,
        "Strat_Sharpe": strat_perf["Sharpe"],
        "BM_CAGR": ew_perf["CAGR"] * 100,
        "BM_Sharpe": ew_perf["Sharpe"],
        "Excess_CAGR": (strat_perf["CAGR"] - ew_perf["CAGR"]) * 100,
        "t_stat": tstat_info["t_stat"],
        "N_months": len(strat_ret),
    })

regime_df_result = pd.DataFrame(regime_results)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*85)
print("레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크 — regime_v4")
print("="*85)
display(regime_df_result.round(3))

# 승패 판정
print("\n" + "-"*60)
print("레짐별 승패 (Sharpe 기준):")
print("-"*60)
for _, row in regime_df_result.iterrows():
    regime = row["Regime"]
    win = "WIN" if row["Strat_Sharpe"] > row["BM_Sharpe"] else "LOSE"
    excess = row["Excess_CAGR"]
    print(f"{regime:22s} | {win:4s} | 초과수익: {excess:+.1f}%p | t={row['t_stat']:.2f}")

logger.info("Cell 5 완료: regime_v4 기반 레짐별 검증")


2026-02-13 14:54:16,600 | INFO | regime_v4: 765 months
2026-02-13 14:54:16,601 | INFO | 레짐 종류: ['Bear', 'Bull', 'Neutral']
2026-02-13 14:54:16,604 | INFO | Bear 내부: {'declining': 151, 'recovering': 19}
2026-02-13 14:54:16,981 | WARNING | Bear(recovering): 데이터 부족 (n=1), 스킵



레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크 — regime_v4


,Regime,Strat_CAGR,Strat_Sharpe,BM_CAGR,BM_Sharpe,Excess_CAGR,t_stat,N_months
0,Bull,19.714,1.273,31.785,2.907,-12.071,2.775,110
1,Neutral,27.129,1.767,0.257,0.015,26.872,NaN,13
2,Bear,25.896,1.036,-32.695,-1.479,58.591,0.181,23
3,Bear(declining),27.163,1.064,-37.005,-1.801,64.167,0.352,22


2026-02-13 14:54:16,989 | INFO | Cell 5 완료: regime_v4 기반 레짐별 검증



------------------------------------------------------------
레짐별 승패 (Sharpe 기준):
------------------------------------------------------------
Bull                   | LOSE | 초과수익: -12.1%p | t=2.77
Neutral                | WIN  | 초과수익: +26.9%p | t=nan
Bear                   | WIN  | 초과수익: +58.6%p | t=0.18
Bear(declining)        | WIN  | 초과수익: +64.2%p | t=0.35


In [8]:
# =============================================================================
# 05_D.ipynb — Cell 6 (regime_v4)
# 목적: regime_v4 기반 레짐별 모멘텀 성과 검증
# 산출물: regime_results, sharpe_pivot
# 주의: A팩터와 동일한 레짐/벤치마크 사용
# =============================================================================

TOP_N = 30  # A팩터와 동일

# ─────────────────────────────────────────────────────────────────────────────
# 1) 레짐 데이터 로드
# ─────────────────────────────────────────────────────────────────────────────
regime_df = pd.read_parquet(PATHS["INTERIM"] / "regime_v4.parquet")

if "date" in regime_df.columns:
    regime_df = regime_df.set_index("date")
regime_df.index = pd.to_datetime(regime_df.index)

# 월말 리샘플링
regime_m = regime_df.resample("ME").last()

logger.info(f"regime_m: {regime_m.shape[0]} months")
logger.info(f"레짐 종류: {regime_m['regime'].unique().tolist()}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 백테스트 함수 (레짐 필터링 버전)
# ─────────────────────────────────────────────────────────────────────────────
def backtest_by_regime(mom_signal: pd.DataFrame, ret_1m: pd.DataFrame,
                       regime_m: pd.DataFrame, regime_col: str, 
                       target_regime: str, top_n: int = 30) -> pd.Series:
    """특정 레짐 기간만 백테스트"""
    port_rets = []
    
    for date in mom_signal.index:
        # 레짐 확인
        if date not in regime_m.index:
            continue
        if regime_m.loc[date, regime_col] != target_regime:
            continue
        if date not in ret_1m.index:
            continue
        
        # 모멘텀 신호
        sig = mom_signal.loc[date].dropna()
        if len(sig) < top_n:
            continue
        
        # 상위 N종목
        top_tickers = sig.nlargest(top_n).index
        
        # 다음 달 수익률
        next_idx = ret_1m.index.get_loc(date)
        if next_idx + 1 >= len(ret_1m):
            continue
        next_date = ret_1m.index[next_idx + 1]
        
        # 포트폴리오 수익률
        port_ret = ret_1m.loc[next_date, top_tickers].mean()
        port_rets.append({"date": next_date, "ret": port_ret})
    
    if not port_rets:
        return pd.Series(dtype=float)
    
    return pd.DataFrame(port_rets).set_index("date")["ret"]

# ─────────────────────────────────────────────────────────────────────────────
# 3) 레짐별 백테스트 (MOM_6_0 고정)
# ─────────────────────────────────────────────────────────────────────────────
mom_signal = mom_signals["MOM_6_0"]  # 전체 기간 1위 전략
REGIME_COL = "regime"

regimes = regime_m[REGIME_COL].dropna().unique().tolist()
regime_results = []

for regime in regimes:
    # 전략 수익률
    strat_ret = backtest_by_regime(mom_signal, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크 (같은 레짐 기간)
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(strat_ret) < 3 or len(ew_sub) < 3:
        logger.warning(f"{regime}: 데이터 부족, 스킵")
        continue
    
    # 성과 계산
    strat_perf = calc_perf(strat_ret)
    ew_perf = calc_perf(ew_sub)
    
    # 초과수익 통계
    common_idx = strat_ret.index.intersection(ew_sub.index)
    if len(common_idx) > 0:
        excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
        tstat_info = calc_tstat(excess)
    else:
        tstat_info = {"t_stat": np.nan, "IR": np.nan, "N": 0}
    
    regime_results.append({
        "Regime": regime,
        "Strat_CAGR": strat_perf["CAGR"] * 100,
        "Strat_Sharpe": strat_perf["Sharpe"],
        "BM_CAGR": ew_perf["CAGR"] * 100,
        "BM_Sharpe": ew_perf["Sharpe"],
        "Excess_CAGR": (strat_perf["CAGR"] - ew_perf["CAGR"]) * 100,
        "t_stat": tstat_info["t_stat"],
        "N_months": len(strat_ret),
    })

regime_df_result = pd.DataFrame(regime_results)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크")
print("="*80)
display(regime_df_result.round(3))

# ─────────────────────────────────────────────────────────────────────────────
# 5) 승패 판정
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*50)
print("레짐별 승패 (Sharpe 기준):")
print("-"*50)

for _, row in regime_df_result.iterrows():
    regime = row["Regime"]
    win = "✓ WIN" if row["Strat_Sharpe"] > row["BM_Sharpe"] else "✗ LOSE"
    excess = row["Excess_CAGR"]
    print(f"{regime:20s} | {win:8s} | 초과수익: {excess:+.1f}%p | t={row['t_stat']:.2f}")

# 승률 계산
wins = (regime_df_result["Strat_Sharpe"] > regime_df_result["BM_Sharpe"]).sum()
total = len(regime_df_result)
print(f"\n승률: {wins}/{total} ({wins/total*100:.0f}%)")

logger.info("Cell 6 완료: regime_v4 기반 레짐별 검증")

2026-02-13 14:54:17,018 | INFO | regime_m: 765 months
2026-02-13 14:54:17,020 | INFO | 레짐 종류: ['Bear', 'Bull', 'Neutral']



레짐별 모멘텀(MOM_6_0) 성과 vs EW 벤치마크


,Regime,Strat_CAGR,Strat_Sharpe,BM_CAGR,BM_Sharpe,Excess_CAGR,t_stat,N_months
0,Bear,27.572,1.032,-32.695,-1.479,60.267,0.147,23
1,Bull,23.761,1.389,31.785,2.907,-8.023,3.208,110
2,Neutral,29.243,1.689,0.257,0.015,28.986,NaN,13


2026-02-13 14:54:17,318 | INFO | Cell 6 완료: regime_v4 기반 레짐별 검증



--------------------------------------------------
레짐별 승패 (Sharpe 기준):
--------------------------------------------------
Bear                 | ✓ WIN    | 초과수익: +60.3%p | t=0.15
Bull                 | ✗ LOSE   | 초과수익: -8.0%p | t=3.21
Neutral              | ✓ WIN    | 초과수익: +29.0%p | t=nan

승률: 2/3 (67%)


In [9]:
# =============================================================================
# 02_D.ipynb — Cell 7
# 목적: 레짐별 최적 Lookback 탐색
# 산출물: lookback_results, lookback_pivot
# 주의: 모든 레짐 × 모든 Lookback 조합 테스트
# =============================================================================

from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
# 1) 설정
# ─────────────────────────────────────────────────────────────────────────────
LOOKBACKS = ["MOM_3_1", "MOM_6_0", "MOM_6_1", "MOM_12_0", "MOM_12_1"]
REGIME_COL = "regime"
regimes = ["Bull", "Neutral", "Bear"]  # Crash 제외 (데이터 없음)

logger.info(f"Lookback 옵션: {LOOKBACKS}")
logger.info(f"레짐 목록: {regimes}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 전체 조합 백테스트
# ─────────────────────────────────────────────────────────────────────────────
lookback_results = []

total_combinations = len(regimes) * len(LOOKBACKS)
pbar = tqdm(total=total_combinations, desc="Lookback × Regime 테스트")

for regime in regimes:
    # EW 벤치마크 (해당 레짐)
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ew_sub) < 3:
        pbar.update(len(LOOKBACKS))
        continue
    
    ew_perf = calc_perf(ew_sub)
    
    for lb_name in LOOKBACKS:
        mom_sig = mom_signals[lb_name]
        strat_ret = backtest_by_regime(mom_sig, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
        
        if len(strat_ret) < 3:
            pbar.update(1)
            continue
        
        strat_perf = calc_perf(strat_ret)
        
        # 초과수익 통계
        common_idx = strat_ret.index.intersection(ew_sub.index)
        if len(common_idx) > 0:
            excess = strat_ret.loc[common_idx] - ew_sub.loc[common_idx]
            excess_cagr = strat_perf["CAGR"] - ew_perf["CAGR"]
        else:
            excess_cagr = np.nan
        
        lookback_results.append({
            "Regime": regime,
            "Lookback": lb_name,
            "Strat_Sharpe": strat_perf["Sharpe"],
            "BM_Sharpe": ew_perf["Sharpe"],
            "Excess_CAGR": excess_cagr * 100,
            "N_months": len(strat_ret),
        })
        
        pbar.update(1)

pbar.close()

lookback_df = pd.DataFrame(lookback_results)

# ─────────────────────────────────────────────────────────────────────────────
# 3) 피벗: 레짐 × Lookback Sharpe
# ─────────────────────────────────────────────────────────────────────────────
sharpe_pivot = lookback_df.pivot(index="Regime", columns="Lookback", values="Strat_Sharpe")
sharpe_pivot = sharpe_pivot[LOOKBACKS].round(3)  # 컬럼 순서 정리

print("\n" + "="*80)
print("레짐 × Lookback: 전략 Sharpe")
print("="*80)
display(sharpe_pivot)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 피벗: 레짐 × Lookback 초과수익
# ─────────────────────────────────────────────────────────────────────────────
excess_pivot = lookback_df.pivot(index="Regime", columns="Lookback", values="Excess_CAGR")
excess_pivot = excess_pivot[LOOKBACKS].round(1)

print("\n" + "="*80)
print("레짐 × Lookback: 초과수익 (%p)")
print("="*80)
display(excess_pivot)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 레짐별 최적 Lookback 추출
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("레짐별 최적 Lookback (Sharpe 기준):")
print("-"*60)

best_lookback = {}
for regime in sharpe_pivot.index:
    row = sharpe_pivot.loc[regime]
    best_lb = row.idxmax()
    best_sharpe = row.max()
    
    # 해당 레짐 BM Sharpe
    bm_sharpe = lookback_df[lookback_df["Regime"] == regime]["BM_Sharpe"].iloc[0]
    
    win = "WIN" if best_sharpe > bm_sharpe else "LOSE"
    diff = best_sharpe - bm_sharpe
    
    best_lookback[regime] = best_lb
    print(f"{regime:20s} -> {best_lb:10s} (Sharpe {best_sharpe:.3f}) | BM {bm_sharpe:.3f} | {win} ({diff:+.3f})")

# ─────────────────────────────────────────────────────────────────────────────
# 6) 최적 Lookback으로도 BM 못 이기는 레짐 표시
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("결론: 레짐별 모멘텀 사용 권장")
print("-"*60)

for regime in sharpe_pivot.index:
    row = sharpe_pivot.loc[regime]
    best_lb = row.idxmax()
    best_sharpe = row.max()
    bm_sharpe = lookback_df[lookback_df["Regime"] == regime]["BM_Sharpe"].iloc[0]
    
    if best_sharpe > bm_sharpe:
        print(f"{regime:20s} -> 사용 O | 최적: {best_lb}")
    else:
        print(f"{regime:20s} -> 사용 X | (최적 {best_lb}로도 BM한테 짐)")

logger.info("Cell 7 완료: 레짐별 Lookback 민감도 테스트")

2026-02-13 14:54:17,378 | INFO | Lookback 옵션: ['MOM_3_1', 'MOM_6_0', 'MOM_6_1', 'MOM_12_0', 'MOM_12_1']


2026-02-13 14:54:17,380 | INFO | 레짐 목록: ['Bull', 'Neutral', 'Bear']
Lookback × Regime 테스트: 100%|██████████| 15/15 [00:01<00:00, 10.95it/s]


레짐 × Lookback: 전략 Sharpe


Lookback,MOM_3_1,MOM_6_0,MOM_6_1,MOM_12_0,MOM_12_1
Regime,,,,,
Bear,0.828,1.032,0.844,1.199,1.248
Bull,1.188,1.389,1.359,1.204,1.064
Neutral,2.517,1.689,1.074,1.954,1.104



레짐 × Lookback: 초과수익 (%p)


Lookback,MOM_3_1,MOM_6_0,MOM_6_1,MOM_12_0,MOM_12_1
Regime,,,,,
Bear,53.8,60.3,58.9,69.2,73.1
Bull,-10.8,-8.0,-9.3,-11.9,-14.2
Neutral,NaN,NaN,NaN,NaN,NaN


2026-02-13 14:54:18,790 | INFO | Cell 7 완료: 레짐별 Lookback 민감도 테스트



------------------------------------------------------------
레짐별 최적 Lookback (Sharpe 기준):
------------------------------------------------------------
Bear                 -> MOM_12_1   (Sharpe 1.248) | BM -1.479 | WIN (+2.727)
Bull                 -> MOM_6_0    (Sharpe 1.389) | BM 2.907 | LOSE (-1.518)
Neutral              -> MOM_3_1    (Sharpe 2.517) | BM 0.015 | WIN (+2.502)

------------------------------------------------------------
결론: 레짐별 모멘텀 사용 권장
------------------------------------------------------------
Bear                 -> 사용 O | 최적: MOM_12_1
Bull                 -> 사용 X | (최적 MOM_6_0로도 BM한테 짐)
Neutral              -> 사용 O | 최적: MOM_3_1


In [10]:
# =============================================================================
# 02_D.ipynb — Cell 8
# 목적: D-3 변동성 조정 모멘텀 (ret / vol)
# 산출물: mom_vol_adjusted, 레짐별 성과 비교
# 주의: 동일 수익률이면 변동성 낮은 놈이 상위
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 변동성 계산 (과거 N개월 수익률의 표준편차)
# ─────────────────────────────────────────────────────────────────────────────
def calc_volatility(ret_1m: pd.DataFrame, lookback: int = 6) -> pd.DataFrame:
    """월간 수익률의 rolling 표준편차 (연환산)"""
    vol = ret_1m.rolling(window=lookback, min_periods=3).std() * np.sqrt(12)
    return vol

# 6개월 변동성
vol_6m = calc_volatility(ret_1m, lookback=6)

logger.info(f"vol_6m: {vol_6m.shape}")
logger.info(f"샘플 (최근 3개월, 5종목):\n{vol_6m.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 변동성 조정 모멘텀: MOM / VOL
# ─────────────────────────────────────────────────────────────────────────────
# 기존 MOM_6_0 사용
mom_raw = mom_signals["MOM_6_0"]

# 변동성 조정 (Sharpe-like ratio)
mom_vol_adj = mom_raw / vol_6m

# inf, -inf 처리
mom_vol_adj = mom_vol_adj.replace([np.inf, -np.inf], np.nan)

logger.info(f"mom_vol_adj 샘플:\n{mom_vol_adj.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 레짐별 백테스트: MOM_6_0 vs MOM_6_0_VOL_ADJ
# ─────────────────────────────────────────────────────────────────────────────
REGIME_COL = "regime"
regimes = [r for r in regime_m[REGIME_COL].dropna().unique()]

comparison_results = []

for regime in regimes:
    # 기존 MOM_6_0
    ret_raw = backtest_by_regime(mom_raw, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 변동성 조정 MOM
    ret_vol = backtest_by_regime(mom_vol_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ret_raw) < 3 or len(ret_vol) < 3:
        continue
    
    perf_raw = calc_perf(ret_raw)
    perf_vol = calc_perf(ret_vol)
    perf_ew = calc_perf(ew_sub)
    
    comparison_results.append({
        "Regime": regime,
        "MOM_6_0_Sharpe": perf_raw["Sharpe"],
        "MOM_VOL_ADJ_Sharpe": perf_vol["Sharpe"],
        "BM_Sharpe": perf_ew["Sharpe"],
        "VOL_ADJ_vs_RAW": perf_vol["Sharpe"] - perf_raw["Sharpe"],
        "N_months": len(ret_raw),
    })

comp_df = pd.DataFrame(comparison_results).round(3)

# ─────────────────────────────────────────────────────────────────────────────
# 4) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("D-3: 변동성 조정 모멘텀 vs 기존 모멘텀")
print("="*80)
display(comp_df)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 개선 여부 판정
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*50)
print("변동성 조정 효과 (VOL_ADJ - RAW):")
print("-"*50)

for _, row in comp_df.iterrows():
    regime = row["Regime"]
    diff = row["VOL_ADJ_vs_RAW"]
    result = "개선" if diff > 0 else "악화"
    print(f"{regime:20s} | {result:4s} ({diff:+.3f})")

improved = (comp_df["VOL_ADJ_vs_RAW"] > 0).sum()
total = len(comp_df)
print(f"\n개선 레짐: {improved}/{total}")

logger.info("Cell 8 완료: D-3 변동성 조정 모멘텀 테스트")

2026-02-13 14:54:18,828 | INFO | vol_6m: (153, 467)
2026-02-13 14:54:18,832 | INFO | 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.311  0.187  0.213  0.187  0.203
2026-01-31  0.307  0.228  0.227  0.237  0.173
2026-02-28  0.297  0.189  0.183  0.222  0.156
2026-02-13 14:54:18,837 | INFO | mom_vol_adj 샘플:
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.507  1.751  1.177 -0.377  0.264
2026-01-31  0.554  1.108  0.871 -0.530  0.671
2026-02-28  0.103  0.954  0.401 -0.657  0.442



D-3: 변동성 조정 모멘텀 vs 기존 모멘텀


,Regime,MOM_6_0_Sharpe,MOM_VOL_ADJ_Sharpe,BM_Sharpe,VOL_ADJ_vs_RAW,N_months
0,Bear,1.032,1.138,-1.479,0.106,23
1,Bull,1.389,1.240,2.907,-0.149,110
2,Neutral,1.689,1.893,0.015,0.204,13


2026-02-13 14:54:19,389 | INFO | Cell 8 완료: D-3 변동성 조정 모멘텀 테스트



--------------------------------------------------
변동성 조정 효과 (VOL_ADJ - RAW):
--------------------------------------------------
Bear                 | 개선   (+0.106)
Bull                 | 악화   (-0.149)
Neutral              | 개선   (+0.204)

개선 레짐: 2/3


In [11]:
# =============================================================================
# 02_D.ipynb — Cell 9
# 목적: D-4 하방 방어형 모멘텀 (조정 시 덜 빠지는 놈)
# 산출물: mom_downside_adj, 레짐별 성과 비교
# 주의: 상승률 대비 하락 방어력 반영
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 1) 하방 변동성 계산 (Downside Deviation)
# ─────────────────────────────────────────────────────────────────────────────
def calc_downside_vol(ret_1m: pd.DataFrame, lookback: int = 6) -> pd.DataFrame:
    """
    하방 변동성: 음수 수익률만으로 계산한 표준편차
    - 상승은 무시, 하락만 봄
    - 조정 시 덜 빠지는 종목 = 하방 변동성 낮음
    """
    # 음수만 남기고 나머지는 0
    neg_ret = ret_1m.clip(upper=0)
    
    # rolling std (연환산)
    downside_vol = neg_ret.rolling(window=lookback, min_periods=3).std() * np.sqrt(12)
    
    return downside_vol

# 6개월 하방 변동성
downside_vol_6m = calc_downside_vol(ret_1m, lookback=6)

logger.info(f"downside_vol_6m: {downside_vol_6m.shape}")
logger.info(f"샘플 (최근 3개월, 5종목):\n{downside_vol_6m.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2) 일반 변동성 vs 하방 변동성 비교
# ─────────────────────────────────────────────────────────────────────────────
print("\n변동성 비교 (최근 시점, 5종목):")
print("-" * 50)
print(f"전체 변동성:\n{vol_6m.iloc[-1, :5].round(3)}")
print(f"하방 변동성:\n{downside_vol_6m.iloc[-1, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3) 하방 조정 모멘텀: MOM / Downside Vol
# ─────────────────────────────────────────────────────────────────────────────
mom_raw = mom_signals["MOM_6_0"]

# 하방 변동성 조정 (Sortino-like ratio)
mom_downside_adj = mom_raw / downside_vol_6m

# inf, -inf 처리
mom_downside_adj = mom_downside_adj.replace([np.inf, -np.inf], np.nan)

logger.info(f"mom_downside_adj 샘플:\n{mom_downside_adj.iloc[-3:, :5].round(3)}")

# ─────────────────────────────────────────────────────────────────────────────
# 4) 레짐별 백테스트: MOM_6_0 vs D-3 vs D-4
# ─────────────────────────────────────────────────────────────────────────────
REGIME_COL = "regime"
regimes = [r for r in regime_m[REGIME_COL].dropna().unique()]

comparison_results = []

for regime in regimes:
    # 기존 MOM_6_0 (D-1)
    ret_raw = backtest_by_regime(mom_raw, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 변동성 조정 (D-3)
    ret_vol = backtest_by_regime(mom_vol_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # 하방 조정 (D-4)
    ret_down = backtest_by_regime(mom_downside_adj, ret_1m, regime_m, REGIME_COL, regime, TOP_N)
    
    # EW 벤치마크
    regime_dates = regime_m[regime_m[REGIME_COL] == regime].index
    ew_sub = ew_ret.loc[ew_ret.index.isin(regime_dates)]
    
    if len(ret_raw) < 3 or len(ret_down) < 3:
        continue
    
    perf_raw = calc_perf(ret_raw)
    perf_vol = calc_perf(ret_vol)
    perf_down = calc_perf(ret_down)
    perf_ew = calc_perf(ew_sub)
    
    comparison_results.append({
        "Regime": regime,
        "D1_MOM_6_0": perf_raw["Sharpe"],
        "D3_VOL_ADJ": perf_vol["Sharpe"],
        "D4_DOWN_ADJ": perf_down["Sharpe"],
        "BM_Sharpe": perf_ew["Sharpe"],
        "Best": max(perf_raw["Sharpe"], perf_vol["Sharpe"], perf_down["Sharpe"]),
        "N_months": len(ret_raw),
    })

comp_df = pd.DataFrame(comparison_results).round(3)

# ─────────────────────────────────────────────────────────────────────────────
# 5) 결과 출력
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("D-1 vs D-3 vs D-4 비교 (Sharpe)")
print("="*80)
display(comp_df)

# ─────────────────────────────────────────────────────────────────────────────
# 6) 레짐별 최적 전략
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "-"*60)
print("레짐별 최적 모멘텀 전략:")
print("-"*60)

for _, row in comp_df.iterrows():
    regime = row["Regime"]
    d1 = row["D1_MOM_6_0"]
    d3 = row["D3_VOL_ADJ"]
    d4 = row["D4_DOWN_ADJ"]
    bm = row["BM_Sharpe"]
    
    # 최고 전략 찾기
    best_val = max(d1, d3, d4)
    if best_val == d1:
        best_name = "D-1"
    elif best_val == d3:
        best_name = "D-3"
    else:
        best_name = "D-4"
    
    # BM 대비 승패
    win = "WIN" if best_val > bm else "LOSE"
    
    print(f"{regime:20s} | 최적: {best_name} ({best_val:.3f}) | BM: {bm:.3f} | {win}")

logger.info("Cell 9 완료: D-4 하방 방어형 모멘텀 테스트")

2026-02-13 14:54:19,429 | INFO | downside_vol_6m: (153, 467)
2026-02-13 14:54:19,433 | INFO | 샘플 (최근 3개월, 5종목):
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  0.157  0.035  0.073  0.120  0.090
2026-01-31  0.158  0.067  0.072  0.176  0.068
2026-02-28  0.154  0.067  0.071  0.176  0.068
2026-02-13 14:54:19,439 | INFO | mom_downside_adj 샘플:
ticker          A   AAPL   ABBV    ABT   ACGL
date                                         
2025-12-31  1.003  9.248  3.439 -0.589  0.596
2026-01-31  1.079  3.771  2.754 -0.713  1.718
2026-02-28  0.198  2.687  1.034 -0.827  1.023



변동성 비교 (최근 시점, 5종목):
--------------------------------------------------
전체 변동성:
ticker
A       0.297
AAPL    0.189
ABBV    0.183
ABT     0.222
ACGL    0.156
Name: 2026-02-28 00:00:00, dtype: float64
하방 변동성:
ticker
A       0.154
AAPL    0.067
ABBV    0.071
ABT     0.176
ACGL    0.068
Name: 2026-02-28 00:00:00, dtype: float64

D-1 vs D-3 vs D-4 비교 (Sharpe)


,Regime,D1_MOM_6_0,D3_VOL_ADJ,D4_DOWN_ADJ,BM_Sharpe,Best,N_months
0,Bear,1.032,1.138,1.169,-1.479,1.169,23
1,Bull,1.389,1.240,1.077,2.907,1.389,110
2,Neutral,1.689,1.893,1.880,0.015,1.893,13


2026-02-13 14:54:20,328 | INFO | Cell 9 완료: D-4 하방 방어형 모멘텀 테스트



------------------------------------------------------------
레짐별 최적 모멘텀 전략:
------------------------------------------------------------
Bear                 | 최적: D-4 (1.169) | BM: -1.479 | WIN
Bull                 | 최적: D-1 (1.389) | BM: 2.907 | LOSE
Neutral              | 최적: D-3 (1.893) | BM: 0.015 | WIN


In [12]:
# =============================================================================
# 05_D.ipynb — Cell 10 (D팩터 최종 결론, regime_v4 재검증)
# =============================================================================
"""
# D팩터 (모멘텀) 최종 결론 — regime_v4 재검증

## 테스트 요약

| 가설 | 내용 | 결과 |
|------|------|------|
| D-1  | 추세 지속 (단순 모멘텀) | 조건부 유효 (Bear) |
| D-2  | 실적 기반 모멘텀 | 생략 (A팩터와 상관 문제) |
| D-3  | 변동성 조정 모멘텀 | 조건부 유효 (Neutral) |
| D-4  | 하방 방어형 모멘텀 | Bear에서 D-3과 거의 동일, 독립 채택 불필요 |

---

## regime_v4 기준 레짐별 MOM_6_0 성과 (셀 5)

  Regime          Sharpe   Excess_CAGR   t_stat   승패
  Bull            1.273    -12.1%p       2.775    LOSE
  Neutral         1.767    +26.9%p       NaN      WIN
  Bear            1.036    +58.6%p       0.181    WIN
  Bear(declining) 1.064    +64.2%p       0.352    WIN
  Bear(recovering) 표본 부족 (n=1)

## 레짐별 Lookback 민감도 — Sharpe (셀 7)

  Lookback    MOM_3_1  MOM_6_0  MOM_6_1  MOM_12_0  MOM_12_1
  Bear        0.828    1.032    0.844    1.199     1.248
  Bull        1.188    1.389    1.359    1.204     1.064
  Neutral     2.517    1.689    1.074    1.954     1.104

  최적: Bear→MOM_12_1(1.248)  Neutral→MOM_3_1(2.517)

## D-3 변동성 조정 효과 (셀 8)

  Regime    D-1      D-3      차이
  Bear      1.032    1.138    +0.106 개선
  Bull      1.389    1.240    -0.149 악화
  Neutral   1.689    1.893    +0.204 개선

## D-1 vs D-3 vs D-4 비교 (셀 9)

  Regime    D-1    D-3    D-4    Best
  Bear      1.032  1.138  1.169  D-4 (≈D-3, 차이 미미)
  Bull      1.389  1.240  1.077  D-1 (LOSE)
  Neutral   1.689  1.893  1.880  D-3

---

## v2 → v4 변경 영향

  v2: Neutral → D-3 (MOM_6_0/VOL)       v4: Neutral → D-3 (MOM_3_1/VOL)
  v2: Contraction → D-1 (MOM_12_1)      v4: Bear → D-1 (MOM_12_1)

  Lookback 변경: Neutral에서 MOM_6_0 → MOM_3_1
  원인: v4 Neutral이 v2보다 좁아 단기 lookback이 적합

---

## A vs D 비교 (regime_v4)

  | 레짐          | A-3               | D 최적            | 권장             |
  |--------------|-------------------|-------------------|-----------------|
  | Bull         | ❌ (-12.2%)       | ❌ (-12.1%)       | EW (팩터 비중 0)  |
  | Neutral      | ⚠ (+0.6%, 미미)  | ✅ D-3 (2.517)    | D 메인           |
  | Bear         | ✅ (+72.6%,t=1.83)| ✅ D-1 (+73.1%)   | 둘 다 비중       |
  |  declining   | ✅ (t=2.21)       | ✅ (t=0.35)       | A 메인 + D 보조  |

---

## 팩터 매핑 확정 (regime_v4)

  | 레짐     | 전략     | Lookback    | Sharpe | 비고               |
  |---------|---------|-------------|--------|---------------------|
  | Bull    | 비활성   | -           | -      | BM한테 짐            |
  | Neutral | D-3     | MOM_3_1/VOL | 2.517  | 변동성조정, n=13 주의 |
  | Bear    | D-1     | MOM_12_1    | 1.248  | 장기모멘텀 방어       |

  ⚠ Bear에서 A-3(t=2.21) > D-1(t=0.35) → Bear 메인은 A-3, D-1은 보조
  ⚠ Neutral n=13 표본 — 03_Multi에서 재확인 필요
"""
print("Cell 10: D팩터 regime_v4 재검증 결론 확정")

Cell 10: D팩터 regime_v4 재검증 결론 확정


In [13]:
from pathlib import Path
import numpy as np
SAVE_DIR = Path(r"C:\QP2\data\interim")

# ── D-1: MOM_12_1 z-score ──
d1_wide = mom_signals["MOM_12_1"]
d1_long = d1_wide.stack().reset_index()
d1_long.columns = ["date", "ticker", "d1_raw"]
d1_long["date"] = pd.to_datetime(d1_long["date"])
d1_long = d1_long.dropna(subset=["d1_raw"])
d1_long["d1_z"] = d1_long.groupby("date")["d1_raw"].transform(
    lambda x: (winsorize(x) - winsorize(x).mean()) / winsorize(x).std()
)

# ── D-3: MOM_3_1 / VOL_6M z-score ──
mom_3_1_wide = mom_signals["MOM_3_1"]
d3_raw_wide = mom_3_1_wide / vol_6m.replace(0, np.nan)
d3_long = d3_raw_wide.stack().reset_index()
d3_long.columns = ["date", "ticker", "d3_raw"]
d3_long["date"] = pd.to_datetime(d3_long["date"])
d3_long = d3_long.dropna(subset=["d3_raw"])
d3_long["d3_z"] = d3_long.groupby("date")["d3_raw"].transform(
    lambda x: (winsorize(x) - winsorize(x).mean()) / winsorize(x).std()
)

# ── 병합 ──
d_signal = d1_long[["date", "ticker", "d1_z"]].merge(
    d3_long[["date", "ticker", "d3_z"]],
    on=["date", "ticker"], how="outer"
)

d_signal.to_parquet(SAVE_DIR / "d_signal.parquet", index=False)
print(f"✅ D 시그널 저장: {SAVE_DIR / 'd_signal.parquet'}")
print(f"   {len(d_signal)} rows, {d_signal['ticker'].nunique()} tickers")
print(f"   D-1 non-null: {d_signal['d1_z'].notna().sum()}")
print(f"   D-3 non-null: {d_signal['d3_z'].notna().sum()}")
# --- 여기까지 복붙 ---

✅ D 시그널 저장: C:\QP2\data\interim\d_signal.parquet
   69834 rows, 467 tickers
   D-1 non-null: 65631
   D-3 non-null: 69834
